
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Appendix_Agent_Deployment_Patterns
#### Duration: 5 minutes

## Overview

This course registers its agent in Unity Catalog and deploys the same registered model as a **Databricks App** (the **agent-airbnb-assistant** app), which is the deployment target Databricks recommends for new agent projects. Deploying to Model Serving endpoints with `agents.deploy()` is still available for existing deployments.

This appendix provides a brief overview of the deployment landscape so you can make informed decisions when deploying your own agents.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">This appendix is reference material only. The evaluation techniques covered in this course apply regardless of which deployment method you choose.</p>
</div>

## A. Model Serving vs. Databricks Apps

| | Model Serving (`agents.deploy()`) | Databricks Apps (recommended) |
|---|---|---|
| **Deploy** | A single function call | Git-based development with bundles, with explicit auth in `databricks.yml` |
| **Infrastructure** | Provisioned automatically, with limited control over the server configuration | Full control over the code and server configuration |
| **Agent code** | Logged as an MLflow artifact | An async FastAPI server (MLflow AgentServer) that you can run locally with `uv run start-app` |
| **Built in** | Inference tables, tracing, and the Review App | A chat UI and an `/invocations` endpoint |

### When to use each

- **Model Serving** works well for quick prototyping and existing deployments that don't need migration yet.
- **Databricks Apps** is the recommended path for new production deployments where you need Git-based CI/CD, local development, and full control over the serving infrastructure.

Both approaches support MLflow tracing and evaluation. The `mlflow.genai.evaluate()` patterns you learned in this course work identically against agents deployed via either method.

For full details, see: Author an AI agent and deploy it on Databricks Apps (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent" target="_blank">AWS</a> | <a href="https://docs.databricks.com/azure/en/generative-ai/agent-framework/author-agent" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-framework/author-agent" target="_blank">GCP</a>)

## B. Migrating from Model Serving to Databricks Apps

If you have an agent deployed via `agents.deploy()` and want to migrate to Databricks Apps, the process involves restructuring your code and configuration.

### Migration steps

1. **Download** the agent's MLflow artifacts.
2. **Restructure** the class methods into decorated functions.
3. **Configure** the app in `databricks.yml`.
4. **Deploy** with `databricks bundle deploy`.

### What changes

| Aspect | Model Serving | Databricks Apps |
|---|---|---|
| **Agent code** | Class-based `ResponsesAgent` with `predict()` / `predict_stream()` | Module-level `@invoke()` and `@stream()` decorated functions |
| **Configuration** | MLmodel file + model config YAML | `databricks.yml` with resource declarations |
| **Execution model** | Synchronous | Async recommended (`async def`) |
| **Deployment command** | `agents.deploy()` | `databricks bundle deploy` |

### What stays the same

- Core agent logic (tool definitions, LLM calls, prompt templates)
- MLflow tracing and evaluation integration
- Unity Catalog tool registration
- Request/response payload format

For the full migration guide, see: Migrate an agent from Model Serving to Databricks Apps (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">AWS</a> | <a href="https://docs.databricks.com/azure/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">GCP</a>)

## C. Agents on Databricks

Databricks offers a spectrum of approaches — from fully managed, no-code options to fully custom, code-first development.

<div style="display: flex; gap: 18px; flex-wrap: wrap; justify-content: center; margin: 20px 0; font-family: sans-serif;">

  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #5A6F77;"></div>
    <div style="display: inline-block; background: #5A6F77; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">LEGACY</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Knowledge Assistant</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Bring your own AI Search index, choose your chunking strategy, and attach multiple source types (up to 10). Full control over the retrieval pipeline.</div>
  </div>

  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">MANAGED</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Genie Agent</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Declarative agents that understand your data. Attach tables for structured queries or UC volumes for document search — Databricks handles tool orchestration and query generation.</div>
  </div>

  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">CODE-FIRST</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Custom Agent</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Write your own agent logic, tools, and orchestration. Use the OpenAI Agents SDK, LangGraph, or any Python framework, then deploy it as a Databricks App, the recommended way to host agents.</div>
  </div>

</div>

**Knowledge Assistants** offer customization (bring your own AI Search index) but are a legacy path. **Genie Agents** are the recommended declarative option for data-aware agents: you declare *what* data or documents your agent should work with, and the platform optimizes *how*. **Custom agents** are code-first: you own the agent logic, tools, and orchestration end to end, and Databricks Apps is the recommended way to host them.

#### The Frontier: Platform Capabilities for Custom Agents

Custom agents don't have to be built from scratch. Databricks is adding platform capabilities that take care of common agent infrastructure for you. Here's what the frontier currently looks like:

<div style="display: flex; gap: 18px; flex-wrap: wrap; justify-content: center; margin: 20px 0 12px 0; font-family: sans-serif;">

  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">NEW · BETA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Agent developer CLI</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Create a “hello world” custom agent in under 10 minutes with a handful of commands, pre-configured with the capabilities below.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/agents/custom-agents/agent-bricks-cli" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/agent-bricks-cli" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/agent-bricks-cli" style="color: #1976d2; text-decoration: underline;">GCP</a></div>
  </div>
  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">NEW · BETA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Managed memory and sessions</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Help agents get the right context at the right time, within a conversation and across sessions.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/agents/custom-agents/stateful-agents" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/stateful-agents" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/stateful-agents" style="color: #1976d2; text-decoration: underline;">GCP</a></div>
  </div>
  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="display: inline-block; background: #FF5F46; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">NEW · BETA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Agentic sandboxes</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Let agents access data and run code in an isolated, secure environment.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/compute/serverless/sandbox" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/sandbox" style="color: #1976d2; text-decoration: underline;">Azure</a></div>
  </div>
  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">GA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Agent hosting</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Host your agent on Databricks serverless infrastructure.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/agents/custom-agents/author-agent" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/custom-agents/author-agent" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/custom-agents/author-agent" style="color: #1976d2; text-decoration: underline;">GCP</a></div>
  </div>
  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">GA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Tracing and evaluation</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">End-to-end tracing and evaluation for agents with MLflow. You will use MLflow tracing throughout this course.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/mlflow3/genai/agent-observability-and-quality" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/agent-observability-and-quality" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/agent-observability-and-quality" style="color: #1976d2; text-decoration: underline;">GCP</a></div>
  </div>
  <div style="flex: 1 1 300px; max-width: 340px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="display: inline-block; background: #00A972; color: #fff; font-size: 11px; font-weight: 700; letter-spacing: 0.06em; padding: 3px 10px; border-radius: 10px; margin-bottom: 8px;">GA</div>
    <div style="font-size: 17px; font-weight: 700; color: #0b2026;">Out of the box tools and MCPs</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Pre-built first- and third-party MCP servers, plus pre-built tools for web search, code execution, and more.</div>
    <div style="font-size: 13px; margin-top: 10px; color: #5A6F77;"><a href="https://docs.databricks.com/aws/en/agents/mcp-tools/" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/mcp-tools/" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/mcp-tools/" style="color: #1976d2; text-decoration: underline;">GCP</a></div>
  </div>

</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Beta features can change and may need to be enabled in your workspace before you can use them. Availability can also differ by cloud.
</p>
</div>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
